« model_19 — SABİT NOKTALAR ÜZERİNDE ÖĞRENİLEN İLİŞKİLER · MATEMATİK · AŞAMA 0: TARİF »

**Soru: model_19'un zemini (yeni parçalar kapalı) toplamayı hangi LR programıyla en iyi öğreniyor?** Tarif burada seçilir, Aşama 1'in bütün kollarında aynen kullanılır.
Tasarım `TASARIM_19.md`, karar kuralları koşudan önce `belge/onkayit/model_19.md`.

| ne | değer |
|---|---|
| veri | model_15 `veri_cok.pt` · 2+3 terim, her terim 0..500 · iz `8d0f89938c67e0e7` |
| biçim | `<eos> soru cevap <eos>` · kayıp YALNIZ cevabın rakamlarında ve EOS'ta |
| model | D 128 (sıra 64 + içerik 64) · 256 hareket, aktif 8, 4 katman · attention 4 × 32 · defter · yeni parçalar KAPALI (zemin) |
| eğitim | Adam, weight decay yok · batch 4096 · 20.000 adım · sabit LR + son %20 (16.000 → 20.000) cosine soğutma |
| ölçüm · kayıt | ölçüm ve ağırlık her 500, tam yedek her 2.000 (her tam yedekte decompose: `decompose/t<N>_math.html`); bitişte tutulan soruların TAMAMI |

| aşama | koşu | değişken |
|---|---|---|
| 0a | `PR_MATH_TRAIN_LR001` · `LR002` · `LR004` · `LR010` | LR tepesi 0,001 / 0,002 / 0,004 / 0,01 (taban lr/10, ısınma yok) |
| 0b | `PR_MATH_TRAIN_WARMUP` | seçilen LR + ilk 200 adım ısınma |
| 0b | `PR_MATH_TRAIN_FLOOR` | seçilen LR + soğutma tabanı 0,0001 |

**İki tür deneme ayrı izlenir** (kullanıcı, 25 Eylül: *"model içinde denediklerimiz ve eğitimde denediklerimiz ayrı şekilde takip etmeliyiz"*): **eğitim denemeleri** `PR_MATH_TRAIN_*` yalnız tarifi değiştirir (model = zemin), **model denemeleri** `PR_MATH_ARCH_*` yalnız modeli değiştirir (tarif = `TARIF`). İkisini birden değiştiren koşu başlamaz; tür ve değişen ayar pakete ve günlüğe yazılır. Bu defter şimdilik yalnız eğitim denemelerini (Aşama 0) koşar; model denemelerinin hücreleri `TARIF` yazılınca eklenir.

**Sıra:** `0 HAZIRLIK` → `1a`–`1d` (0a koşuları, her biri kendi hücresinde) → `4 NABIZ` · `5 EĞRİ` → hepsi bitince `6 0a KARARI` → HAZIRLIK'ta `LR_0A`'yı yaz, `0 HAZIRLIK`'ı yeniden çalıştır → `2a` · `2b` → `7 SONUÇ`
**Çekirdek düşerse:** `0 HAZIRLIK` → `3 SÜRDÜR`

**Dönen hücre YOK.** Koşu arka planda bir iplikte döner, her hücre hemen geri gelir (kural 8). **Koşuları kullanıcı başlatır (kural 0).**

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre, sonra "3 SURDUR".
import os, sys, subprocess

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/model_19'
DATA_FILE = '/content/drive/MyDrive/model_15/veri_cok.pt'
os.makedirs(ROOT, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
REPO = '/content/sekerai'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/sekerahmet/sekerai.git', REPO], check=True)
SRC = REPO + '/deneme2/model_19'
CODE = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('kod   ' + CODE)
# Yerel commit GitHub'a gitmediyse burada durur, ESKI kodla kosmaz.
assert os.path.exists(SRC + '/train_19.py'), 'depoda model_19 YOK -- yerelde git push gerekli'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for _m in ('model_19', 'train_19', 'data_mat_19', 'decompose_19', 'diagnose_19', 'data_stories_19'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import model_19
import train_19 as train
import data_mat_19 as data

# Kural 9: veri Drive'dan; iz yuklerken YENIDEN hesaplanip karsilastirilir.
train_q, heldout_q = data.load(DATA_FILE, 'cok')
TRAIN = data.make_windows(train_q)
METRIC = data.make_metric(train_q, heldout_q, device='cuda', limit=2000)

# Onkayit (belge/onkayit/model_19.md): batch 4096, 20.000 adim, son %20 sogutma, olcum/agirlik 500, tam yedek 2.000.
COMMON = dict(batch=4096, steps=20000, eval_every=500, save_every=2000, weights_every=500, seed=0,
              decay_start=16000, decay_floor=0.1)
# D 128 (kullanici: "deneme olarak 128 yaparız"): sira 64 + icerik 64; bas x boyut = D.  rank 256 >= D: tam matrisler.
MATH = dict(d_order=64, d_content=64, vectors=256, active=8, layers=4, attn_heads=4, attn_dim=32, t_max=64, rank=256)
BASE = dict(embed=False, readout=False, chain_sim=False, distance=False, attn_after=(0,))      # zemin
MODEL_BASE = dict(MATH, **BASE)

# IKI TUR DENEME AYRI IZLENIR.  Kullanici, 25 Eylul: "model içinde denediklerimiz ve eğitimde denediklerimiz ayrı
# şekilde takip etmeliyiz".  EGITIM denemesi yalniz tarifi degistirir (model = zemin); MODEL denemesi yalniz modeli
# degistirir (tarif = TARIF).  Ikisini birden degistiren kosu BASLAMAZ; tur ve degisen ayar pakete ve gunluge yazilir.
REF_RECIPE = dict(lr=0.002, warmup=0, decay_floor=0.1)    # egitim denemelerinin olculdugu referans tarif
LR_0A = None       # "6 0a KARARI"nin sonucu buraya yazilir, sonra bu hucre yeniden calistirilir
TARIF = None       # Asama 0 bitince: dict(lr=..., warmup=..., decay_floor=...) -- model denemelerinin tarifi

TRAIN_TRIALS = {'PR_MATH_TRAIN_LR%03d' % round(lr * 1000): dict(lr=lr) for lr in (0.001, 0.002, 0.004, 0.01)}
if LR_0A is not None:
    TRAIN_TRIALS['PR_MATH_TRAIN_WARMUP'] = dict(lr=LR_0A, warmup=200)
    if abs(LR_0A - 0.001) > 1e-12:        # LR 0,001'de 0,0001 taban zaten lr/10: kosulmaz (onkayit)
        TRAIN_TRIALS['PR_MATH_TRAIN_FLOOR'] = dict(lr=LR_0A, decay_floor=1e-4 / LR_0A)
ARCH_TRIALS = {}
if TARIF is not None:
    for part, kw in (('BASE', {}), ('E', dict(embed=True)), ('RPC', dict(readout=True)),
                     ('DIST', dict(distance=True)), ('ATT2', dict(attn_after=(0, 2)))):
        for s in (0, 1, 2):
            ARCH_TRIALS['PR_MATH_ARCH_%s_S%d' % (part, s)] = (s, dict(MODEL_BASE, **kw))

# ad -> (tur, egitim ayari, model ayari)
CONFIGS = {**{n: ('egitim', tr, MODEL_BASE) for n, tr in TRAIN_TRIALS.items()},
           **{n: ('model', dict(TARIF, seed=s), mk) for n, (s, mk) in ARCH_TRIALS.items()}}
EXTRA = dict(data='model_15 veri_cok.pt', fingerprint=data.FINGERPRINTS['cok'], exam='MAT', code=CODE,
             onkayit='belge/onkayit/model_19.md')


def trial(name):
    '''-> (tur, referansa gore degisen).  Ikisini birden degistiren deneme DURUR.'''
    kind, tr, mk = CONFIGS[name]
    recipe = {k: tr.get(k, REF_RECIPE[k]) for k in REF_RECIPE}
    model = {k: v for k, v in mk.items() if MODEL_BASE.get(k) != v}
    if kind == 'egitim':
        assert not model, name + ': egitim denemesi modeli degistiremez -- ' + str(model)
        return kind, {k: v for k, v in recipe.items() if REF_RECIPE[k] != v}
    assert recipe == {k: TARIF.get(k, REF_RECIPE[k]) for k in REF_RECIPE}, name + ': model denemesi tarifi degistiremez'
    return kind, model


def memory_gb(d_order, d_content, vectors, active, layers, attn_heads, **_):
    '''Ileri gecisin geri yayilim icin SAKLADIGI, TAHMIN (olculmedi): katman basina uzaklik tablosu, aktif
    hareketler ve nokta; attention agirliklari; defterin (T,T) benzerligi.'''
    B, T, d = COMMON['batch'], TRAIN[0].shape[1], d_order + d_content
    return B * T * (layers * (vectors + active * d + 3 * d) + 2 * attn_heads * T + 2 * T) * 4 / 1e9


def launch(name, resume=None, steps=None):
    '''CONFIGS[name] ile arka planda baslatir (kural 8); GPU kapisi hucrenin kendisinde.'''
    kind, tr, mk = CONFIGS[name]
    kind, changed = trial(name)
    kw = dict(COMMON, **tr)
    if steps is not None:
        kw['steps'] = steps
    return train.start(name, TRAIN, data.N, metric=METRIC, device='cuda', root=ROOT, vocab=data.VOCAB, resume=resume,
                       extra=dict(EXTRA, trial=kind, changed=changed), **kw, **mk)


n = TRAIN[0].shape[0]
print('veri  %s   iz %s   kapi GECTI' % (os.path.basename(DATA_FILE), data.FINGERPRINTS['cok']))
print('egitim %s soru   tutulan %s soru   pencere T=%d' % (f'{len(train_q):,}', f'{len(heldout_q):,}', TRAIN[0].shape[1]))
print('1 epok = %.0f adim   %s adim = %.0f epok' % (n / COMMON['batch'], f"{COMMON['steps']:,}",
                                                   COMMON['steps'] * COMMON['batch'] / n))
for kind_title, kind_key in (('EGITIM DENEMELERI (model = zemin)', 'egitim'), ('MODEL DENEMELERI (tarif = TARIF)', 'model')):
    names = [x for x in CONFIGS if CONFIGS[x][0] == kind_key]
    print('\n' + kind_title + ('' if names else '  -- henuz yok'))
    for name in names:
        _m = model_19.PointRelation(data.N, **CONFIGS[name][2])
        print('  %-24s degisen %-34s parametre %s   saklanan ~%.2f GB'
              % (name, trial(name)[1], f'{sum(p.numel() for p in _m.parameters()):,}', memory_gb(**CONFIGS[name][2])))

In [ ]:
# 1a KOSU -- 0a, LR 0,001  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_MATH_TRAIN_LR001'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
assert NAME in CONFIGS, NAME + ' CONFIGS\'te yok -- 0b icin once HAZIRLIK\'ta LR_0A yazilmali'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "4 NABIZ".
print(launch(NAME))

In [ ]:
# 1b KOSU -- 0a, LR 0,002  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_MATH_TRAIN_LR002'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
assert NAME in CONFIGS, NAME + ' CONFIGS\'te yok -- 0b icin once HAZIRLIK\'ta LR_0A yazilmali'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "4 NABIZ".
print(launch(NAME))

In [ ]:
# 1c KOSU -- 0a, LR 0,004  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_MATH_TRAIN_LR004'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
assert NAME in CONFIGS, NAME + ' CONFIGS\'te yok -- 0b icin once HAZIRLIK\'ta LR_0A yazilmali'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "4 NABIZ".
print(launch(NAME))

In [ ]:
# 1d KOSU -- 0a, LR 0,01  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_MATH_TRAIN_LR010'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
assert NAME in CONFIGS, NAME + ' CONFIGS\'te yok -- 0b icin once HAZIRLIK\'ta LR_0A yazilmali'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "4 NABIZ".
print(launch(NAME))

In [ ]:
# 2a KOSU -- 0b, ISINMA (secilen LR + ilk 200 adim)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_MATH_TRAIN_WARMUP'
# Yalniz "6 0a KARARI"ndan sonra: HAZIRLIK'ta LR_0A yazili olmali.
# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
assert NAME in CONFIGS, NAME + ' CONFIGS\'te yok -- 0b icin once HAZIRLIK\'ta LR_0A yazilmali'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "4 NABIZ".
print(launch(NAME))

In [ ]:
# 2b KOSU -- 0b, TABAN 0,0001 (secilen LR)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_MATH_TRAIN_FLOOR'
# Yalniz "6 0a KARARI"ndan sonra; LR_0A 0,001 ise bu kosu YOK (taban zaten lr/10).
# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
assert NAME in CONFIGS, NAME + ' CONFIGS\'te yok -- 0b icin once HAZIRLIK\'ta LR_0A yazilmali'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "4 NABIZ".
print(launch(NAME))

In [ ]:
# 3 SURDUR  |  GPU  |  tekrar: GUVENLI
# Cekirdek dustuyse: once "0 HAZIRLIK", sonra BU hucre.  NAME'i sec.
# Kural 1: uzatma SURDURMEDIR -- uzatmak icin STEPS'i buyut, bu hucreyi calistir.
import os, re, torch
NAME = 'PR_MATH_TRAIN_LR002'     # CONFIGS'teki adlardan biri
STEPS = 20000                   # hedef; uzatmada buyut (kural 1)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, 2.5 * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

_d = ROOT + '/' + NAME
_n = sorted((int(re.findall('[0-9]+', f)[0]), f) for f in os.listdir(_d) if re.match('t[0-9]+[.]pt$', f))
assert _n, 'surdurme paketi YOK -- ' + _d
RESUME_FROM = _d + '/' + _n[-1][1]
print('son nokta  %s   adim %s   hedef %s' % (RESUME_FROM, f'{_n[-1][0]:,}', f'{STEPS:,}'))
assert _n[-1][0] < STEPS, 'zaten hedefe varmis -- uzatmak icin STEPS buyut'
print(launch(NAME, resume=RESUME_FROM, steps=STEPS))

In [ ]:
# 4 NABIZ  |  CPU  |  tekrar: GUVENLI
# DONMEZ, hemen doner.  HICBIR SEY KOSTURMAZ (kural 8) -- yalniz gunlugu basar.
train.show_log(30)

In [ ]:
# 5 EGRI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar, hicbir sey kosturmaz
# Ust: heldout ve train birebir dogru (her 500 adim, 2.000 + 2.000 soru).  Alt: HER ADIMIN kaybi.
import os
import torch
import matplotlib.pyplot as plt


def log_points(name):
    '''gunluk.txt -> {adim: (kayip, train, heldout)}.'''
    r, path = {}, ROOT + '/' + name + '/gunluk.txt'
    if os.path.exists(path):
        for s in open(path, encoding='utf-8'):
            p = s.split()
            if len(p) >= 6 and p[1].isdigit():
                try:
                    r[int(p[1])] = tuple(float(x) for x in p[2:5])
                except ValueError:
                    pass
    return r


def step_losses(name):
    '''Canli kosudan (train.RUNS) ya da diskteki son paketten.'''
    k = train.RUNS[name].result.get('step_losses') if name in train.RUNS else None
    path = ROOT + '/model_' + name + '.pt'
    if k is None and os.path.exists(path):
        try:
            k = torch.load(path, weights_only=False, map_location='cpu').get('step_losses')
        except Exception as h:          # yazilirken okunduysa
            print('  %s paketi okunamadi (%s) -- tekrar dene' % (name, h))
    return None if k is None else k[~k.isnan()]


fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for name in CONFIGS:
    r, k = log_points(name), step_losses(name)
    if r:
        x = sorted(r)
        a1.plot(x, [r[i][2] for i in x], label=name + ' heldout')
        a1.plot(x, [r[i][1] for i in x], ':', label=name + ' train')
        s = max(x, key=lambda i: r[i][2])
        print('%-22s %d nokta   son adim %s  heldout %.4f   en iyi %.4f (adim %s)'
              % (name, len(x), f'{x[-1]:,}', r[x[-1]][2], r[s][2], f'{s:,}'))
    if k is not None and len(k) > 200:
        a2.plot(k.numpy(), lw=0.4, label=name)
a1.set_ylabel('birebir dogru cevap')
a1.legend(fontsize=7, ncol=2)
a1.grid(alpha=0.3)
a2.set_yscale('log')
a2.set_ylabel('her adimin kaybi')
a2.set_xlabel('adim')
a2.grid(alpha=0.3)
plt.show()

In [ ]:
# 6 0a KARARI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar; kural belge/onkayit/model_19.md'den, aynen
# Bitisteki tam olcumde heldout birebir dogru en yuksek; 1 puandan kucuk farklar esit, esitlerde kucuk LR.
# Iraksayan elenir: kayip NaN ya da son 2.000 adimin ortalama kaybi ilk 500 adiminkinden buyuk.
import math, os, re, torch

rows = []
for name, (kind, tr, mk) in CONFIGS.items():
    if not name.startswith('PR_MATH_TRAIN_LR'):
        continue
    path = ROOT + '/' + name + '/gunluk.txt'
    done = [s for s in open(path, encoding='utf-8')] if os.path.exists(path) else []
    end = [s for s in done if ' BITTI ' in s]
    if not end:
        print('%-22s BITMEDI -- karar icin dort kosunun da bitmesi gerekir' % name)
        continue
    heldout = float(re.search(r'heldout ([0-9.]+)', end[-1]).group(1))
    k = torch.load(ROOT + '/model_' + name + '.pt', weights_only=False, map_location='cpu').get('step_losses')
    k = k[~k.isnan()]
    diverged = (not math.isfinite(float(k[-1]))) or float(k[-2000:].mean()) > float(k[:500].mean())
    rows.append((name, tr['lr'], heldout, diverged))
    print('%-22s LR %.3f   heldout (tam) %.4f   %s' % (name, tr['lr'], heldout, 'IRAKSADI -- elendi' if diverged else ''))

ok = [r for r in rows if not r[3]]
if len(rows) == 4 and ok:
    best = max(r[2] for r in ok)
    tied = sorted((r for r in ok if best - r[2] < 0.01), key=lambda r: r[1])
    print('\nen yuksek %.4f; 1 puan icinde: %s' % (best, ', '.join('%s (%.4f)' % (r[0], r[2]) for r in tied)))
    print('KURALIN SECTIGI LR: %s  ->  HAZIRLIK hucresinde LR_0A = %s yaz, HAZIRLIK\'i yeniden calistir'
          % (tied[0][1], tied[0][1]))

In [ ]:
# 7 SONUC  |  GPU  |  tekrar: GUVENLI -- kosu BITTIKTEN sonra, tutulan sorularin TAMAMI
NAME = 'PR_MATH_TRAIN_LR002'

# --- GPU KAPISI (CLAUDE.md kural 2)
import random
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
print('GPU kapisi GECTI: ' + torch.cuda.get_device_name(0))
assert not (NAME in train.RUNS and train.RUNS[NAME].alive), NAME + ' HALA KOSUYOR -- egitilen model olculmez'

k = torch.load(ROOT + '/model_' + NAME + '.pt', weights_only=False, map_location='cuda')
m = model_19.PointRelation.from_package(k).cuda().eval()
print('%s   adim %s   heldout (tam) %.4f' % (NAME, f"{k['step']:,}", k['heldout_acc']))
for measure, title in (('terim', 'KAC TERIMLI'), ('hane', 'CEVAP KAC HANELI')):
    t = data.breakdown(m, heldout_q, device='cuda', measure=measure)
    print(title)
    for a, x in t.items():
        print('%6s  heldout accuracy %.4f   first_digit %.4f   length_ok %.4f   n %d'
              % (a, x['accuracy'], x['first_digit'], x['length_ok'], x['n']))
print('\nBASAMAK (tutulan)')
data.digit_table(data.digit_check(m, heldout_q, device='cuda'))
print('\nGOZLE  (tutulandan 12 soru, tohum 7)')
data.show(m, random.Random(7).sample(heldout_q, 12), device='cuda')
print('\nDECOMPOSE sayfalari (Drive, tarayicida acilir):')
_d = ROOT + '/' + NAME + '/decompose'
for f in sorted(os.listdir(_d)) if os.path.isdir(_d) else []:
    if f.endswith('.html'):
        print('  ' + _d + '/' + f)

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki adimda CIKMADAN ONCE Drive'a kaydeder.
train.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for name in CONFIGS:
    _d = ROOT + '/' + name
    if not os.path.isdir(_d):
        print('%-22s henuz kayit yok' % name)
        continue
    _f = sorted(os.listdir(_d))
    _b = sum(os.path.getsize(_d + '/' + f) for f in _f if os.path.isfile(_d + '/' + f))
    print('%-22s %3d yedek   %.3f GB   %s' % (name, sum(f.endswith('.pt') for f in _f), _b / 1e9, _d))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi